In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# What SMOTE does – very simple
# Imagine you have a dataset where one group is very small compared to another.
# Example: 10 “sick” people vs 90 “healthy” people.
# If you train a model on this, it will mostly predict “healthy” because that’s most of your data.
# SMOTE fixes this by making fake “sick” examples that are similar to the real ones, so your model has more balanced data to learn from.

In [44]:
df = pd.read_csv('cs-training.csv', index_col=0)

print("Before cleaning:", df.shape)

Before cleaning: (150000, 11)


In [45]:
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [46]:
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

In [47]:
# Remove extreme outliers
df = df[df['age'] > 18]
df = df[df['DebtRatio'] < 5000]
df = df[df['RevolvingUtilizationOfUnsecuredLines'] < 2]

In [48]:
print("After cleaning:", df.shape)

After cleaning: (148151, 11)


In [49]:
# Create new features (feature engineering)
df['TotalLatePayments'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] + 
                            df['NumberOfTime60-89DaysPastDueNotWorse'] + 
                            df['NumberOfTimes90DaysLate'])

df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

In [50]:
print("New columns added:")
print(df[['TotalLatePayments', 'IncomePerDependent']].head())
print("\nTotal columns now:", df.shape[1])
# Should be 13 columns (11 original + 2 new)

New columns added:
   TotalLatePayments  IncomePerDependent
1                  2              3040.0
2                  0              1300.0
3                  2              3042.0
4                  0              3300.0
5                  1             63588.0

Total columns now: 13


In [34]:
# Separate features and target
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

In [35]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [36]:
# Fix class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)


In [37]:

print("After SMOTE:", X_train_sm.shape)
print(pd.Series(y_train_sm).value_counts())

After SMOTE: (221262, 12)
SeriousDlqin2yrs
0    110631
1    110631
Name: count, dtype: int64
